## 1. torch_prep_kfold.py --initial_split  → writes *_trn_final.csv, *_tst_preprocess.csv

In [66]:
import sys
import logging
import argparse
import numpy as np
import pandas as pd
from typing import Any, Optional, List, Tuple
from sklearn.model_selection import GroupKFold, StratifiedGroupKFold
import os

In [67]:
random_state = 42
test_percentage = 0.15
np.random.seed(42)

In [68]:
###############################################################################
# Data Loading (Reference + Features)
###############################################################################

In [69]:
id_col = 'sequence'
label_col = 'bind_avg'
df1 = pd.read_csv('exp_data_all.csv')
ref_data = df1[[id_col, label_col]].copy()

print(ref_data)

                                 sequence  bind_avg
0    GAGGAAGCAGCCCTCGCCCCTGTCGGTGGAAAGAAG -0.758634
1    GCAGCCGAGGCGGAGAGAGAGAGAGGACAGCTTACG -1.003319
2    ATCTGATCAAAACAACGAATTCCAAAACAAAGTAAT -0.800322
3    CCAATATTCCTTTGTGAGACCCTCCACAAATGCTAA -0.941242
4    GAGGACGCGAACCGGCACGCTGCGCCTTTAAGGAGT -0.684116
..                                    ...       ...
163  ACATAGGGACGGGGCCATGCGGTGGGCGGGTGGAAC  1.074979
164  GAAAACCAGCGAGACCGCATGGTCTCACTTATAAGT  1.152711
165  CGCGGAGACCCGAAGCACGTGGTATCCATACTAGTT  2.121860
166  GCCCCCGACCCCGCGCACGCGGCCCCGCCCCGCGCG  1.105868
167  ATTAGCCAAACTAAACACGTGTATTGATTTTAGATG  1.884407

[168 rows x 2 columns]


In [70]:
usecols = ['sequence','run','VDWAALS','EEL','EGB','ESURF','HB Energy','Hydrophobic Energy','Pi-Pi Energy','Delta_Entropy']

df2 = pd.read_csv('rawdat.csv', usecols=usecols)
feature_data = df2.copy()

print(feature_data)

                                    sequence  run  VDWAALS       EEL  \
0       GCGCTGGGAGGCGAACACGTGCCCGCCGCCCCATCC    9 -252.110 -1886.830   
1       GCGCTGGGAGGCGAACACGTGCCCGCCGCCCCATCC    9 -238.510 -1881.424   
2       GCGCTGGGAGGCGAACACGTGCCCGCCGCCCCATCC    9 -246.721 -1895.687   
3       GCGCTGGGAGGCGAACACGTGCCCGCCGCCCCATCC    9 -235.671 -1857.573   
4       GCGCTGGGAGGCGAACACGTGCCCGCCGCCCCATCC    9 -230.214 -1897.268   
...                                      ...  ...      ...       ...   
272155  TTTTTTTTTTTTTTGAGAAAATGAAGACAATTATCT   20 -148.291 -1941.978   
272156  TTTTTTTTTTTTTTGAGAAAATGAAGACAATTATCT   20 -142.375 -1959.709   
272157  TTTTTTTTTTTTTTGAGAAAATGAAGACAATTATCT   20 -167.433 -1911.650   
272158  TTTTTTTTTTTTTTGAGAAAATGAAGACAATTATCT   20 -136.663 -1920.439   
272159  TTTTTTTTTTTTTTGAGAAAATGAAGACAATTATCT   20 -146.999 -1929.330   

             EGB   ESURF  HB Energy  Hydrophobic Energy  Pi-Pi Energy  \
0       1841.253 -36.482  -1.940432         -165.447020 -1.655

In [71]:
search_feat_row = feature_data[(feature_data['sequence'] == 'CGGCTTTTTCTTGAACACGTGGAATATACTAGCGCT') & (feature_data['VDWAALS'] == -207.264)]
print(search_feat_row)

                                    sequence  run  VDWAALS      EEL      EGB  \
105492  CGGCTTTTTCTTGAACACGTGGAATATACTAGCGCT    7 -207.264 -1958.57  1908.59   

         ESURF  HB Energy  Hydrophobic Energy  Pi-Pi Energy  Delta_Entropy  
105492 -34.442  -3.478696         -140.468722     -5.839134     -22.871906  


In [72]:
df_merged = pd.merge(feature_data, ref_data, on=id_col, how="inner")
print(df_merged.head(), df_merged.shape)

                               sequence  run  VDWAALS       EEL       EGB  \
0  GCGCTGGGAGGCGAACACGTGCCCGCCGCCCCATCC    9 -252.110 -1886.830  1841.253   
1  GCGCTGGGAGGCGAACACGTGCCCGCCGCCCCATCC    9 -238.510 -1881.424  1835.847   
2  GCGCTGGGAGGCGAACACGTGCCCGCCGCCCCATCC    9 -246.721 -1895.687  1851.589   
3  GCGCTGGGAGGCGAACACGTGCCCGCCGCCCCATCC    9 -235.671 -1857.573  1814.002   
4  GCGCTGGGAGGCGAACACGTGCCCGCCGCCCCATCC    9 -230.214 -1897.268  1847.934   

    ESURF  HB Energy  Hydrophobic Energy  Pi-Pi Energy  Delta_Entropy  \
0 -36.482  -1.940432         -165.447020 -1.655191e-03     -26.046553   
1 -36.023  -2.003962         -155.422935 -4.708262e-02     -24.150637   
2 -35.802  -2.269901         -142.386371 -5.901517e-29     -24.329875   
3 -34.799  -2.838678         -147.918585 -3.236084e-07     -23.615145   
4 -34.391  -2.810414         -151.012478 -1.784784e-05     -23.698348   

   bind_avg  
0  1.531336  
1  1.531336  
2  1.531336  
3  1.531336  
4  1.531336   (272160, 11)


In [73]:
search_one_row = df_merged[(df_merged['sequence'] == 'CGGCTTTTTCTTGAACACGTGGAATATACTAGCGCT') & (df_merged['VDWAALS'] == -207.264)]
print(search_one_row)


                                    sequence  run  VDWAALS      EEL      EGB  \
105492  CGGCTTTTTCTTGAACACGTGGAATATACTAGCGCT    7 -207.264 -1958.57  1908.59   

         ESURF  HB Energy  Hydrophobic Energy  Pi-Pi Energy  Delta_Entropy  \
105492 -34.442  -3.478696         -140.468722     -5.839134     -22.871906   

        bind_avg  
105492   1.97632  


In [74]:
# Shuffle everything
df_merged = df_merged.sample(frac=1.0, random_state=random_state).reset_index(drop=True)
print(df_merged.head(), df_merged.shape)

                               sequence  run  VDWAALS       EEL       EGB  \
0  CGGCTTTTTCTTGAACACGTGGAATATACTAGCGCT    7 -207.264 -1958.570  1908.590   
1  TCCTAAACAGGAAGCCATGAGGTGAGCAGAGACACT    2 -229.177 -1917.175  1870.986   
2  TTAGAAAAATAGTTTAAAATCTAGAGTTAATTAACC    3 -197.506 -1830.441  1787.958   
3  CCAGCTCTCCACCGCCGCGTGCGCCTGCAGACGCTC    1 -207.553 -1891.794  1845.707   
4  CCCCCAGCGCTCCGGCACGCGCCGGGAGACCTCCGG   19 -201.484 -1923.635  1874.505   

    ESURF  HB Energy  Hydrophobic Energy  Pi-Pi Energy  Delta_Entropy  \
0 -34.442  -3.478696         -140.468722     -5.839134     -22.871906   
1 -32.215 -18.599640         -143.435504     -1.459304     -22.335776   
2 -29.223  -2.514019         -120.507243     -4.043271     -18.798981   
3 -31.909 -13.343306         -138.470452     -3.535617     -22.011054   
4 -31.059  -4.628190         -138.103395     -0.006414     -22.886912   

   bind_avg  
0  1.976320  
1 -0.061198  
2  0.002694  
3  0.153350  
4 -0.483023   (272160, 11)


In [75]:
# Split train vs test by sequence or stratified group
## for regression, not bin or mclass

from sklearn.model_selection import GroupKFold

unique_seqs = df_merged[id_col].unique() #unique sequences are extracted
np.random.seed(random_state)
np.random.shuffle(unique_seqs) #randomly shuffles unique sequences

n_train = int((1 - test_percentage) * len(unique_seqs)) # Compute train/test split boundary

train_seqs = unique_seqs[:n_train] # First 85% (after shuffle) = training sequence IDs.
test_seqs  = unique_seqs[n_train:]

# Filter the rows accordingly
df_train = df_merged[df_merged[id_col].isin(train_seqs)].copy()
df_test  = df_merged[df_merged[id_col].isin(test_seqs)].copy()

print(df_train.shape, df_test.shape)


(230040, 11) (42120, 11)


In [76]:
if "run" in df_train.columns:
    df_train.drop(columns=["run"], inplace=True, errors="ignore")
if "run" in df_test.columns:
    df_test.drop(columns=["run"], inplace=True, errors="ignore")

print(df_train.shape, df_test.shape)

(230040, 10) (42120, 10)


In [77]:
# Save
train_file = f"reg_trn_final.csv"
test_file  = f"reg_tst_preprocess.csv"

df_train.to_csv(train_file, index=False)
df_test.to_csv(test_file, index=False)

In [78]:
###########################################################################
# 2) PROCESS MODE: "train" or "test"
###########################################################################

In [79]:
# Keep only the last X% if requested
keep_last_percent = 90

"""
Retain only the last keep_percent fraction of rows in each sequence group.
If 'run' column exists, sort by it first.
"""

if "run" in df.columns:
    df_sorted = df.sort_values([seq_col, "run"], kind="mergesort")
else:
    df_sorted = df.copy()
group_sizes = df_sorted.groupby(seq_col)[seq_col].transform("size")
cumcount = df_sorted.groupby(seq_col).cumcount()
n_keep = (group_sizes * (keep_percent / 100.0)).astype(int)
n_keep = n_keep.mask(n_keep < 1, 1)  # ensure at least 1 row if fraction>0
mask = cumcount >= (group_sizes - n_keep)
return df_sorted[mask].reset_index(drop=True)

df_train = keep_last_n_percent(df_train, id_col, keep_last_percent)

NameError: name 'seq_col' is not defined